In [3]:
import pandas as pd

In [3]:
rnamigo2_robin_df = pd.read_csv('./outputs/rnamigos2_robin_tpp_results.csv')

In [ ]:
# get the max number in column dock and print the smiles
dock_values = rnamigo2_robin_df['dock']
print(max(dock_values))

# get the smiles of the max number
max_smiles = rnamigo2_robin_df[rnamigo2_robin_df['dock'] == max(dock_values)]['smiles']
print(max_smiles)
#

0.21273175
24082    [H][C@@]12C(N[C@@]3([H])C(N[C@@]4([H])C(N[C@](...
Name: smiles, dtype: object


## Optimizer Comparison Results

Analysis of 4 different molecular optimization methods across 5 random seeds:
- **Graph GA**: Graph-based genetic algorithm
- **SMILES GA**: SMILES-based genetic algorithm  
- **REINVENT**: Deep learning-based generative model
- **Screening**: Random baseline (no optimization)

Each optimizer ran for 1500 oracle evaluations on the RNAmigos2 oracle targeting TPP riboswitch (2gdi).

In [ ]:
import yaml
import numpy as np

# Define the optimizers and their metrics files
optimizers = ['screening', 'graph_ga', 'smiles_ga', 'reinvent', 'gp_bo']
seeds = [0, 1, 2, 3, 5]

# Read all metrics
all_metrics = {}
for optimizer in optimizers:
    optimizer_data = []
    for seed in seeds:
        file_path = f'opt_results/{optimizer}/metrics_{optimizer}_rnamigos2_oracle_{seed}.yaml'
        with open(file_path, 'r') as f:
            metrics = yaml.safe_load(f)
            optimizer_data.append(metrics)
    all_metrics[optimizer] = optimizer_data

print("Successfully loaded metrics for all optimizers!")
print(f"Metrics per run: {list(all_metrics['graph_ga'][0].keys())}")

Successfully loaded metrics for all optimizers!
Metrics per run: ['avg_top1', 'avg_top10', 'avg_top100', 'auc_top1', 'auc_top10', 'auc_top100', 'avg_sa', 'diversity_top100', 'n_oracle']


In [ ]:
# Calculate mean and std for each metric (excluding n_oracle)
metric_names = [k for k in all_metrics['graph_ga'][0].keys() if k != 'n_oracle']

results = {}
for optimizer in optimizers:
    optimizer_results = {}
    for metric in metric_names:
        values = [run[metric] for run in all_metrics[optimizer]]
        mean = np.mean(values)
        std = np.std(values, ddof=1)  # Use sample std (n-1)
        optimizer_results[metric] = f"{mean:.3f} ± {std:.3f}"
    results[optimizer] = optimizer_results

# Create DataFrame with optimizers as columns and metrics as rows
results_df = pd.DataFrame(results)
results_df = results_df.T  # Transpose so optimizers are rows
results_df.index.name = 'Optimizer'

print("Metrics Summary (Mean ± Std across 5 seeds):")
print("=" * 80)
results_df

Metrics Summary (Mean ± Std across 5 seeds):


,avg_top1,avg_top10,avg_top100,auc_top1,auc_top10,auc_top100,avg_sa,diversity_top100
Optimizer,,,,,,,,
screening,-0.029 ± 0.051,-0.091 ± 0.014,-0.157 ± 0.005,-0.072 ± 0.013,-0.119 ± 0.007,-0.190 ± 0.003,2.425 ± 0.060,0.867 ± 0.005
graph_ga,0.546 ± 0.105,0.477 ± 0.080,0.315 ± 0.067,0.257 ± 0.050,0.183 ± 0.049,0.044 ± 0.040,4.505 ± 0.381,0.724 ± 0.022
smiles_ga,0.597 ± 0.721,0.481 ± 0.574,0.367 ± 0.472,0.181 ± 0.271,0.114 ± 0.223,0.009 ± 0.170,6.136 ± 0.509,0.761 ± 0.045
reinvent,0.016 ± 0.072,-0.053 ± 0.028,-0.142 ± 0.017,-0.034 ± 0.035,-0.094 ± 0.013,-0.181 ± 0.008,3.657 ± 0.115,0.884 ± 0.007
gp_bo,0.237 ± 0.077,0.164 ± 0.045,0.073 ± 0.047,0.162 ± 0.064,0.088 ± 0.037,-0.020 ± 0.035,3.132 ± 0.138,0.752 ± 0.045


In [32]:
# Create the final table with metrics as rows and optimizers as columns
final_results = {}
for optimizer in optimizers:
    optimizer_stats = {}
    for metric in metric_names:
        values = [run[metric] for run in all_metrics[optimizer]]
        mean = np.mean(values)
        std = np.std(values, ddof=1)
        optimizer_stats[metric] = f"{mean:.3f} ± {std:.3f}"
    final_results[optimizer] = optimizer_stats

# Create DataFrame with metrics as rows and optimizers as columns
metrics_table = pd.DataFrame(final_results)
metrics_table.index.name = 'Metric'

# Rename columns for better readability
metrics_table.columns = ['Screening', 'Graph GA', 'SMILES GA', 'REINVENT', 'GP BO']

print("\n" + "="*100)
print("OPTIMIZATION RESULTS: Mean ± Std across 5 random seeds (0, 1, 2, 3, 5)")
print("="*100)
print(metrics_table)
print("\n")


OPTIMIZATION RESULTS: Mean ± Std across 5 random seeds (0, 1, 2, 3, 5)
                       Screening       Graph GA      SMILES GA  \
Metric                                                           
avg_top1          -0.029 ± 0.051  0.546 ± 0.105  0.597 ± 0.721   
avg_top10         -0.091 ± 0.014  0.477 ± 0.080  0.481 ± 0.574   
avg_top100        -0.157 ± 0.005  0.315 ± 0.067  0.367 ± 0.472   
auc_top1          -0.072 ± 0.013  0.257 ± 0.050  0.181 ± 0.271   
auc_top10         -0.119 ± 0.007  0.183 ± 0.049  0.114 ± 0.223   
auc_top100        -0.190 ± 0.003  0.044 ± 0.040  0.009 ± 0.170   
avg_sa             2.425 ± 0.060  4.505 ± 0.381  6.136 ± 0.509   
diversity_top100   0.867 ± 0.005  0.724 ± 0.022  0.761 ± 0.045   

                        REINVENT           GP BO  
Metric                                            
avg_top1           0.016 ± 0.072   0.237 ± 0.077  
avg_top10         -0.053 ± 0.028   0.164 ± 0.045  
avg_top100        -0.142 ± 0.017   0.073 ± 0.047  
auc_top1    

In [33]:
# Display the styled table
metrics_table

,Screening,Graph GA,SMILES GA,REINVENT,GP BO
Metric,,,,,
avg_top1,-0.029 ± 0.051,0.546 ± 0.105,0.597 ± 0.721,0.016 ± 0.072,0.237 ± 0.077
avg_top10,-0.091 ± 0.014,0.477 ± 0.080,0.481 ± 0.574,-0.053 ± 0.028,0.164 ± 0.045
avg_top100,-0.157 ± 0.005,0.315 ± 0.067,0.367 ± 0.472,-0.142 ± 0.017,0.073 ± 0.047
auc_top1,-0.072 ± 0.013,0.257 ± 0.050,0.181 ± 0.271,-0.034 ± 0.035,0.162 ± 0.064
auc_top10,-0.119 ± 0.007,0.183 ± 0.049,0.114 ± 0.223,-0.094 ± 0.013,0.088 ± 0.037
auc_top100,-0.190 ± 0.003,0.044 ± 0.040,0.009 ± 0.170,-0.181 ± 0.008,-0.020 ± 0.035
avg_sa,2.425 ± 0.060,4.505 ± 0.381,6.136 ± 0.509,3.657 ± 0.115,3.132 ± 0.138
diversity_top100,0.867 ± 0.005,0.724 ± 0.022,0.761 ± 0.045,0.884 ± 0.007,0.752 ± 0.045


In [34]:
# save the metric table
metrics_table.to_csv("optimizers_metrics.csv")

### Summary

The **metrics table** (`metrics_table`) contains mean ± std for each metric across 5 random seeds.

**Key Metrics Explained:**
- `avg_top1/10/100`: Average RNAmigos2 score of the top 1/10/100 molecules found
- `auc_top1/10/100`: Area under the curve for top scoring molecules over optimization time
- `avg_sa`: Average Synthetic Accessibility score (2-10 scale, lower = easier to synthesize)
- `diversity_top100`: Tanimoto diversity among top 100 molecules (0-1, higher = more diverse)